# Multi-Agent Routing Example

This example demonstrates the handoffs/routing pattern from the OpenAI Agents SDK.

The triage agent receives messages and routes to language-specific agents (French, Spanish, English).

**Source:** Based on `examples/agent_patterns/routing.py` from openai-agents-python

## Setup

Import required dependencies and configure environment.

In [1]:
import os
import uuid
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if OpenAI API key is set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found. Please set it in .env file or environment"
    )

# Redis connection (for future use)
REDIS_URL = os.getenv("REDIS_URL", "redis://redis:6379")
print(f"✅ Environment configured")
print(f"   Redis URL: {REDIS_URL}")

✅ Environment configured
   Redis URL: redis://redis:6379


In [2]:
from openai.types.responses import ResponseContentPartDoneEvent, ResponseTextDeltaEvent
from agents import Agent, RawResponsesStreamEvent, Runner, TResponseInputItem, trace

print("✅ OpenAI Agents SDK imported successfully")

✅ OpenAI Agents SDK imported successfully


## Define Language-Specific Agents

Create three agents, each speaking only one language.

In [3]:
french_agent = Agent(
    name="french_agent",
    instructions="You only speak French",
)

spanish_agent = Agent(
    name="spanish_agent",
    instructions="You only speak Spanish",
)

english_agent = Agent(
    name="english_agent",
    instructions="You only speak English",
)

print("✅ Language agents created:")
print(f"   - {french_agent.name}")
print(f"   - {spanish_agent.name}")
print(f"   - {english_agent.name}")

✅ Language agents created:
   - french_agent
   - spanish_agent
   - english_agent


## Define Triage Agent

The triage agent routes requests to the appropriate language agent.

In [4]:
triage_agent = Agent(
    name="triage_agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[french_agent, spanish_agent, english_agent],
)

print("✅ Triage agent created")
print(f"   Handoffs available: {[h.name for h in triage_agent.handoffs]}")

✅ Triage agent created
   Handoffs available: ['french_agent', 'spanish_agent', 'english_agent']


## Run Conversation

Interact with the agents through streaming responses.

In [5]:
# Create a unique conversation ID for tracing
conversation_id = str(uuid.uuid4().hex[:16])
print(f"Conversation ID: {conversation_id}")

Conversation ID: dcbf77def7eb469f


In [6]:
# Start conversation
user_message = "Bonjour! Comment allez-vous?"  # French: "Hello! How are you?"
print(f"User: {user_message}\n")

agent = triage_agent
inputs: list[TResponseInputItem] = [{"content": user_message, "role": "user"}]

# Run agent with streaming
with trace("Routing example", group_id=conversation_id):
    result = Runner.run_streamed(agent, input=inputs)
    
    print("Assistant: ", end="")
    async for event in result.stream_events():
        if not isinstance(event, RawResponsesStreamEvent):
            continue
        data = event.data
        if isinstance(data, ResponseTextDeltaEvent):
            print(data.delta, end="", flush=True)
        elif isinstance(data, ResponseContentPartDoneEvent):
            print("\n")

print(f"\n✅ Current agent: {result.current_agent.name}")

User: Bonjour! Comment allez-vous?

Assistant: 

Bonjour

 !

 Je

 vais

...

-vous

 aujourd

’hui

 ?


✅ Current agent: french_agent


## Session Persistence with Redis

The conversations above are stored only in memory and are lost when the kernel restarts. 

Let's demonstrate how to use `AgentSession` to persist conversations in Redis, enabling:
- 💾 **Persistence across restarts** - conversations survive kernel/server restarts
- 🔄 **Resume conversations** - continue where you left off
- 👥 **Multi-user support** - separate conversations per user
- 🔍 **Conversation history** - query and analyze past interactions

### Step 1: Create Session and Run Conversation

Start a new conversation with session tracking enabled.

In [7]:
from redis_openai_agents import AgentSession

# Create a new session with automatic conversation ID generation
session = AgentSession.create(user_id="demo_user", redis_url=REDIS_URL)

print(f"✅ Session created")
print(f"   User ID: {session.user_id}")
print(f"   Conversation ID: {session.conversation_id}")

✅ Session created
   User ID: demo_user
   Conversation ID: 4464bb5115034c80


In [8]:
# Run a conversation with the triage agent
user_msg_1 = "Bonjour! Comment allez-vous?"
print(f"User: {user_msg_1}\n")

result_1 = Runner.run_streamed(
    triage_agent,
    input=[{"content": user_msg_1, "role": "user"}]
)

# Stream the response
print("Assistant: ", end="")
async for event in result_1.stream_events():
    if isinstance(event, RawResponsesStreamEvent):
        data = event.data
        if isinstance(data, ResponseTextDeltaEvent):
            print(data.delta, end="", flush=True)

print(f"\n\n✅ Agent routed to: {result_1.current_agent.name}")

# Store the conversation in the session
session.store_agent_result(result_1)
print(f"✅ Conversation stored in Redis")
print(f"   Messages in session: {session.message_count()}")

User: Bonjour! Comment allez-vous?

Assistant: 

10:51:41 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


10:51:42 httpx INFO   HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


10:51:42 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Bonjour

...

 allez

-vous

 ?



✅ Agent routed to: french_agent


✅ Conversation stored in Redis
   Messages in session: 2


### Step 2: Inspect Persisted Data with RedisVL CLI

Use the `rvl` CLI tool to inspect the data stored in Redis.

In [9]:
!rvl index listall -u $REDIS_URL

10:51:43 [RedisVL] INFO   Using Redis address from environment variable, REDIS_URL
10:51:43 [RedisVL] INFO   Indices:
10:51:43 [RedisVL] INFO   1. test_hybrid_filter
10:51:43 [RedisVL] INFO   2. agent_sessions


In [10]:
!rvl index info -i agent_sessions -u $REDIS_URL

10:51:44 [RedisVL] INFO   Using Redis address from environment variable, REDIS_URL


Index Information:
╭────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬╮
│ Index Name         │ Storage Type       │ Prefixes           │ Index Options      │ Indexing           │
├────────────────────┼────────────────────┼────────────────────┼────────────────────┼────────────────────┼┤
| agent_sessions     | HASH               | ['agent_sessions'] | []                 | 0                  |
╰────────────────────┴────────────────────┴────────────────────┴────────────────────┴────────────────────┴╯
Index Fields:
╭──────────────┬──────────────┬──────────────┬──────────────┬──────────────┬╮
│ Name         │ Attribute    │ Type         │ Field Option │ Option Value │
├──────────────┼──────────────┼──────────────┼──────────────┼──────────────┼┤
│ role         │ role         │ TAG          │ SEPARATOR    │ ,            │
│ content      │ content     

      │ SEPARATOR    │ ,            │
│ timestamp    │ timestamp    │ NUMERIC      │              │              │
│ session_tag  │ session_tag  │ TAG          │ SEPARATOR    │ ,            │
│ metadata     │ metadata     │ TEXT         │ WEIGHT       │ 1            │
╰──────────────┴──────────────┴──────────────┴──────────────┴──────────────┴╯


In [11]:
!rvl stats -i agent_sessions -u $REDIS_URL

10:51:44 [RedisVL] INFO   Using Redis address from environment variable, REDIS_URL

Statistics:
╭─────────────────────────────┬────────────╮
│ Stat Key                    │ Value      │
├─────────────────────────────┼────────────┤
│ num_docs                    │ 2          │
│ num_terms                   │ 11         │
│ max_doc_id                  │ 2          │
│ num_records                 │ 21         │
│ percent_indexed             │ 1          │
│ hash_indexing_failures      │ 0          │
│ number_of_uses              │ 3          │
│ bytes_per_record_avg        │ 72.1904754 │
│ doc_table_size_mb           │ 0.01554393 │
│ inverted_sz_mb              │ 0.00144577 │
│ key_table_size_mb           │ 8.29696655 │
│ offset_bits_per_record_avg  │ 8          │
│ offset_vectors_sz_mb        │ 1.52587890 │
│ offsets_per_term_avg        │ 0.76190477 │
│ records_per_doc_avg         │ 10.5       │
│ sortable_values_size_mb     │ 0          │
│ total_indexing_time         │ 0.22599999 │
│ to

### Step 3: Simulate Restart - Clear Python State

Delete all Python variables to simulate a kernel restart or server shutdown. The conversation data remains in Redis.

In [12]:
# Save the conversation ID before deleting session
saved_conversation_id = session.conversation_id
print(f"Conversation ID to restore: {saved_conversation_id}")

# Simulate restart by deleting Python objects
del session, result_1

print("✅ Python state cleared (simulating restart)")
print("   session object no longer exists in memory")

Conversation ID to restore: 4464bb5115034c80
✅ Python state cleared (simulating restart)
   session object no longer exists in memory


### Step 4: Load Session from Redis

Load the conversation from Redis using the conversation ID.

In [13]:
# Load the session from Redis
loaded_session = AgentSession.load(
    conversation_id=saved_conversation_id,
    user_id="demo_user",
    redis_url=REDIS_URL
)

print(f"✅ Session loaded from Redis")
print(f"   Conversation ID: {loaded_session.conversation_id}")
print(f"   Messages restored: {loaded_session.message_count()}")

# Get conversation history in OpenAI Agents SDK format
history = loaded_session.to_agent_inputs()
print(f"\n📜 Conversation history:")
for i, msg in enumerate(history, 1):
    role = msg["role"]
    content = msg["content"][:60] + "..." if len(msg["content"]) > 60 else msg["content"]
    print(f"   {i}. [{role}]: {content}")

10:51:44 redisvl.index.index INFO   Index already exists, not overwriting.


✅ Session loaded from Redis
   Conversation ID: 4464bb5115034c80
   Messages restored: 2

📜 Conversation history:
   1. [user]: Bonjour! Comment allez-vous?
   2. [assistant]: Bonjour ! Je vais très bien, merci. Et vous, comment allez-v...


### Step 5: Continue Conversation with Restored Context

Continue the conversation where we left off, using the loaded history.

In [14]:
# Add a new message to continue the conversation
user_msg_2 = "Parlez-moi de Paris"
print(f"User: {user_msg_2}\n")

# Combine history with new message
continued_inputs = history + [{"content": user_msg_2, "role": "user"}]

# Continue with the french_agent (we know from history which agent to use)
result_2 = Runner.run_streamed(
    french_agent,
    input=continued_inputs
)

# Stream the response
print("Assistant: ", end="")
async for event in result_2.stream_events():
    if isinstance(event, RawResponsesStreamEvent):
        data = event.data
        if isinstance(data, ResponseTextDeltaEvent):
            print(data.delta, end="", flush=True)

print(f"\n\n✅ Conversation continued successfully!")

# Store the new exchange
loaded_session.store_agent_result(result_2)
print(f"✅ New exchange stored in Redis")
print(f"   Total messages in session: {loaded_session.message_count()}")

User: Parlez-moi de Paris

Assistant: 

10:51:45 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Bien

 sûr

 !

...

 depuis

 des

 siècles

.



✅ Conversation continued successfully!
✅ New exchange stored in Redis
   Total messages in session: 6


### Verify Complete Conversation History

View the full conversation history after the continuation.

In [15]:
# View the complete conversation
all_messages = loaded_session.get_messages()

print(f"📜 Complete conversation history ({len(all_messages)} messages):\n")
for i, msg in enumerate(all_messages, 1):
    role = msg["role"]
    content = msg["content"]
    print(f"{i}. [{role.upper()}]:")
    # Truncate long messages for display
    if len(content) > 100:
        print(f"   {content[:100]}...")
    else:
        print(f"   {content}")
    print()

📜 Complete conversation history (6 messages):

1. [USER]:
   Bonjour! Comment allez-vous?

2. [ASSISTANT]:
   Bonjour ! Je vais très bien, merci. Et vous, comment allez-vous ?

3. [USER]:
   Bonjour! Comment allez-vous?

4. [ASSISTANT]:
   Bonjour ! Je vais très bien, merci. Et vous, comment allez-vous ?

5. [USER]:
   Parlez-moi de Paris

6. [ASSISTANT]:
   Bien sûr ! Paris, surnommée « la Ville Lumière », est la capitale de la France. Elle se trouve dans ...



## Try Different Languages

Test routing to Spanish agent.

In [16]:
# New conversation - Spanish
conversation_id_2 = str(uuid.uuid4().hex[:16])
user_message_spanish = "¡Hola! ¿Cómo estás?"  # Spanish: "Hello! How are you?"
print(f"User: {user_message_spanish}\n")

agent = triage_agent
inputs_spanish: list[TResponseInputItem] = [{"content": user_message_spanish, "role": "user"}]

with trace("Routing example - Spanish", group_id=conversation_id_2):
    result = Runner.run_streamed(agent, input=inputs_spanish)
    
    print("Assistant: ", end="")
    async for event in result.stream_events():
        if not isinstance(event, RawResponsesStreamEvent):
            continue
        data = event.data
        if isinstance(data, ResponseTextDeltaEvent):
            print(data.delta, end="", flush=True)
        elif isinstance(data, ResponseContentPartDoneEvent):
            print("\n")

print(f"\n✅ Current agent: {result.current_agent.name}")

User: ¡Hola! ¿Cómo estás?

Assistant: 

10:51:50 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


10:51:51 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


¡

Hola

...

 puedo

 ayudarte

 hoy

?


✅ Current agent: spanish_agent


## Summary

This notebook demonstrated:

### Core Multi-Agent Patterns
1. ✅ **Multiple agents** with different capabilities (language-specific)
2. ✅ **Agent handoffs** - triage agent routes to appropriate agent
3. ✅ **Streaming responses** - real-time token streaming
4. ✅ **Conversation continuity** - maintaining context across turns
5. ✅ **Tracing** - tracking conversations with unique IDs

### Redis Session Persistence ⭐
6. ✅ **Session creation** - create persistent sessions with automatic IDs
7. ✅ **Automatic message storage** - `store_agent_result()` captures all messages
8. ✅ **Conversation persistence** - data survives restarts and lives in Redis
9. ✅ **Session loading** - restore conversations by conversation ID
10. ✅ **History continuation** - seamlessly resume conversations
11. ✅ **Redis inspection** - use `rvl` CLI to inspect persisted data

**Key Takeaways:**
- ✅ Sessions persist across Python restarts (notebook kernel, server restarts, etc.)
- ✅ Automatic message capture from OpenAI Agents SDK responses
- ✅ Handles agent handoffs transparently (unwraps context messages)
- ✅ Built on RedisVL's MessageHistory for reliability

**Next Steps:**
- Implement semantic caching to reduce LLM calls
- Add vector search for knowledge base integration
- Create multi-user conversation management UI